<a href="https://colab.research.google.com/github/hyeonuj0213/20128-/blob/main/20128%EC%9E%90%EC%9C%A8%EC%A3%BC%EC%A0%9C%ED%83%90%EA%B5%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 필요한 라이브러리 설치 (코랩 환경용)
!pip install konlpy tensorflow pandas numpy scikit-learn

import pandas as pd
import numpy as np
from konlpy.tag import Okt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
from google.colab import files

# 2. 데이터 불러오기
uploaded = files.upload()
file_name = list(uploaded.keys())[0]      # 업로드한 실제 파일명 추출
df = pd.read_excel(file_name)             # xlsx 파일이므로 read_excel 사용

df = df.dropna(subset=['document', 'label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f"데이터 개수: {len(df)}개")
print(df['label'].value_counts())

Saving research_dataset_600.xlsx to research_dataset_600 (1).xlsx
데이터 개수: 600개
label
1    300
0    300
Name: count, dtype: int64


In [ ]:
# 3. KoNLPy를 이용한 형태소 분석 및 불용어(Stopwords) 제거
okt = Okt()
stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다', '임', '음']

def preprocess_text(text):
    # 형태소 분석 및 어간 추출
    tokenized = okt.morphs(text, stem=True)
    # 불용어 제거
    tokenized = [word for word in tokenized if not word in stopwords]
    return tokenized

print("텍스트 전처리를 시작합니다. (시간이 조금 걸릴 수 있습니다.)")
df['tokenized'] = df['document'].apply(preprocess_text)

# 4. 정수 인코딩 및 시퀀스 패딩 (단어를 숫자로 변환)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['tokenized'])
vocab_size = len(tokenizer.word_index) + 1 # 단어 집합의 크기

X = tokenizer.texts_to_sequences(df['tokenized'])

# 리뷰의 최대 길이를 확인하여 패딩(길이 맞춤) 처리
max_len = 30
X_pad = pad_sequences(X, maxlen=max_len)
y = np.array(df['label'])

# 5. 학습 데이터와 테스트 데이터 분리 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42)


텍스트 전처리를 시작합니다. (시간이 조금 걸릴 수 있습니다.)


In [ ]:
# 6. LSTM 딥러닝 모델 구축
model = Sequential()
model.add(Embedding(vocab_size, 100)) # 단어를 100차원 벡터로 임베딩
model.add(LSTM(128))                  # LSTM 레이어 (은닉 상태 크기 128)
model.add(Dense(1, activation='sigmoid')) # 이진 분류를 위한 출력층

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

# 7. 모델 학습
print("\n--- 모델 학습을 시작합니다 ---")
history = model.fit(X_train, y_train, epochs=1, batch_size=32, validation_split=0.2)

# 8. 모델 성능 평가
print(model.evaluate(X_test,y_test))
# 소수점 이하 넷째자리까지 출력
print(f'accuracy: {model.evaluate(X_test,y_test)[1]: .4}')


--- 모델 학습을 시작합니다 ---
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 133ms/step - accuracy: 0.7188 - loss: 0.6532 - val_accuracy: 0.7917 - val_loss: 0.5463
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7167 - loss: 0.5754 
[0.5753805041313171, 0.7166666388511658]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7167 - loss: 0.5754
accuracy:  0.7167


In [ ]:
# 2. 데이터 불러오기
uploaded2 = files.upload()
file_name_clean = list(uploaded.keys())[0]      # 업로드한 실제 파일명 추출
clean_df = pd.read_excel(file_name)             # xlsx 파일이므로 read_excel 사용

clean_df = clean_df.dropna(subset=['document', 'label']).reset_index(drop=True)
clean_df['label'] = df['label'].astype(int)

print(f"데이터 개수: {len(df)}개")
print(df['label'].value_counts())

print(f"AI 정제 데이터 개수: {len(clean_df)}개")

# 2. 전처리 생략 후 바로 Keras 토크나이저 적용 (띄어쓰기 기준 단어 분리)
# 이미 문장이 깔끔하게 교정되었으므로, 복잡한 형태소 분석 없이 바로 토큰화합니다.
ai_tokenizer = Tokenizer()
ai_tokenizer.fit_on_texts(clean_df['document'])
ai_vocab_size = len(ai_tokenizer.word_index) + 1 # B 그룹용 단어 사전 크기

X_clean = ai_tokenizer.texts_to_sequences(clean_df['document'])

# 3. 시퀀스 패딩 (길이 맞춤)
max_len_clean = 30
X_clean_pad = pad_sequences(X_clean, maxlen=max_len_clean)
y_clean = np.array(clean_df['label'])

# 4. 학습 데이터와 테스트 데이터 분리 (변수명 겹치지 않게 분리)
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean_pad, y_clean, test_size=0.2, random_state=42
)

# 5. B 그룹용 새로운 LSTM 딥러닝 모델 구축
ai_model = Sequential()
ai_model.add(Embedding(ai_vocab_size, 100))
ai_model.add(LSTM(128))
ai_model.add(Dense(1, activation='sigmoid'))

ai_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

# 6. 모델 학습
print("\n--- AI 정제 데이터 모델 학습 시작 ---")
ai_history = ai_model.fit(X_train_clean, y_train_clean, epochs=1, batch_size=32, validation_split=0.2)

# 7. 모델 성능 평가
print(ai_model.evaluate(X_test_clean,y_test_clean))
# 소수점 이하 넷째자리까지 출력
print(f'accuracy: {ai_model.evaluate(X_test_clean,y_test_clean)[1]: .4}')

Saving cleansed_dataset_final.xlsx to cleansed_dataset_final (3).xlsx
데이터 개수: 600개
label
1    300
0    300
Name: count, dtype: int64
AI 정제 데이터 개수: 600개

--- AI 정제 데이터 모델 학습 시작 ---
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - accuracy: 0.6771 - loss: 0.6668 - val_accuracy: 1.0000 - val_loss: 0.6035
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9583 - loss: 0.6082 
[0.6082465052604675, 0.9583333134651184]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9583 - loss: 0.6082
accuracy:  0.9583
